# SFT stage - Preferred-FT of GPT-2 small on hh-rlhf (`harmless-base`)

Produces the anchor checkpoint for the DPO-SAE project: GPT-2 small finetuned with a plain
LM objective on the `chosen` completions only, so that hh-rlhf's `Human:`/`Assistant:`
format is inside the model's distribution before DPO starts.

The checkpoint is not an intermediate. It is the DPO initialization, the frozen reference
inside the DPO loss, and the model the SAE is trained on.

**Run order:** mount Drive, set secrets, then run the cells top to bottom for the full
~21k-pair run (20-30 min on a T4).

## 1. Drive

First, before anything else. Checkpoints go to Drive, not to the VM disk - a disconnect
partway through an unmounted run loses every checkpoint.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip -q install -U transformers datasets accelerate wandb

## 2. Secrets and environment

The key comes from the Colab secrets panel (the key icon in the left sidebar), never from a
cell. A pasted key gets saved into the notebook's stored output; so does an interactive
`wandb.login()`. Add a secret named `WANDB_API_KEY` and give this notebook access to it.

In [ ]:
import os

from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "DPO-SAE"     # shared with the later DPO and SAE runs
os.environ["WANDB_LOG_MODEL"] = "false"     # checkpoints belong on Drive, not in wandb
os.environ["WANDB_WATCH"] = "false"

# Optional: park the HF cache on Drive so gpt2 and harmless-base survive a fresh VM.
# Both are small, so this is a convenience, not a requirement.
# os.environ["HF_HOME"] = "/content/drive/MyDrive/DPO-SAE/hf-cache"

## 3. Configuration

Two things here are load-bearing and easy to break.

**The budget is counted in pairs, not steps or epochs.** Step count moves with batch size;
the pair budget does not. A running counter drives the stopping condition, the checkpoint
cadence and the validation cadence. `max_steps` and `save_steps` do not.

**Half the split, sampled.** hh-rlhf is not shuffled in a way that makes a prefix
representative, so the half is drawn with a seeded shuffle. The other half is reserved for
DPO: training both stages on the same rows means the DPO reference has already memorized the
`chosen` completions, which shrinks the logprob gap the DPO loss works on. `DATA_SEED` and
the row range are written to the run config and to a manifest so the DPO stage can
reconstruct exactly which half this model saw.

In [ ]:
import torch

MODEL_NAME = "gpt2"           # GPT-2 small, 124M
DATASET = "Anthropic/hh-rlhf"
DATA_DIR = "harmless-base"

DATA_SEED = 1337              # the DPO stage needs this to reconstruct the split
SFT_HALF = 0                  # 0 = first half of the shuffle is SFT's, second half is DPO's

MAX_LEN = 512                 # GPT-2's context is 1024; 512 is this run's budget
MIN_COMPLETION_TOKENS = 16    # a prompt leaving less room than this is dropped
VAL_PAIRS = 512               # fixed val slice, carved from the SFT half
MASK_PROMPT_LOSS = False      # False = plain LM loss over (prompt + chosen), logged either way

CKPT_EVERY_PAIRS = 4_000      # -> 5 checkpoints + final over ~21k pairs
PER_DEVICE_BS = 8
GRAD_ACCUM = 4                # effective batch 32
LR = 5e-5
WARMUP_RATIO = 0.03
LOG_EVERY_STEPS = 10
N_SAMPLE_GENERATIONS = 3

RUN_NAME = "sft-gpt2-hh-21k"
EPOCHS = 1

DRIVE_ROOT = "/content/drive/MyDrive/DPO-SAE"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"       # the artifacts: weights + tokenizer per 4k pairs
FINAL_DIR = f"{RUN_DIR}/final"
RESUME_DIR = f"{RUN_DIR}/_resume"         # rolling Trainer state, disconnect insurance only

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESUME_DIR, exist_ok=True)

os.environ["WANDB_RUN_ID"] = RUN_NAME
os.environ["WANDB_RESUME"] = "allow"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[env] device={DEVICE} run={RUN_NAME}")
print(f"[env] checkpoints -> {CKPT_DIR}")

## 4. Preprocessing

`chosen` and `rejected` are complete dialogues sharing every turn except the final assistant
response, so the split point is the **last** `"\n\nAssistant:"`. Everything up to and
including the marker is the prompt; everything after is the completion.

Rows are dropped when the marker is absent, when `chosen` and `rejected` do not actually
share the resulting prefix (a handful of rows have diverging earlier turns), when the
completion is empty, or when the prompt alone eats the context budget.

Filtering runs **after** sampling, so the usable count lands slightly under the nominal 21k.
The before/after counts are printed at every stage: a sudden drop in yield means the
splitter broke.

In [ ]:
MARKER = "\n\nAssistant:"


def split_dialogue(example):
    r"""Split an hh-rlhf row into (prompt, completion) at the final turn marker.

    Rows look like:
        "\n\nHuman: how do I ...\n\nAssistant: well ...\n\nHuman: ok\n\nAssistant: sure"

    Unusable rows come back empty rather than raising, and keep_row drops them.
    `rejected` is read only to verify the shared prefix - the preference signal it
    carries belongs to DPO, not to this stage.
    """
    chosen, rejected = example["chosen"], example["rejected"]

    idx = chosen.rfind(MARKER)
    if idx == -1:
        return {"prompt": "", "completion": ""}

    prompt = chosen[: idx + len(MARKER)]

    # Guard: the two must actually share the prefix.
    if not rejected.startswith(prompt):
        return {"prompt": "", "completion": ""}

    return {"prompt": prompt, "completion": chosen[idx + len(MARKER):]}


def keep_row(example):
    return len(example["prompt"]) > 0 and len(example["completion"].strip()) > 0

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET, data_dir=DATA_DIR, split="train")
RAW_N = len(raw)

# Seeded shuffle, then halve. Not the first 21k rows: the split is not ordered in a way
# that makes a prefix representative.
shuffled = raw.shuffle(seed=DATA_SEED)
mid = RAW_N // 2
SFT_ROWS = (0, mid) if SFT_HALF == 0 else (mid, RAW_N)
DPO_ROWS = (mid, RAW_N) if SFT_HALF == 0 else (0, mid)

stage_ds = shuffled.select(range(*SFT_ROWS))

print(f"[data] {DATA_DIR} train: {RAW_N} rows")
print(f"[data] shuffle(seed={DATA_SEED}) -> SFT half = rows {SFT_ROWS}, "
      f"{DPO_ROWS} reserved for DPO")
print(f"[data] sampled {len(stage_ds)} pairs for this stage")

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token     # GPT-2 ships no pad token


def encode(example):
    """Tokenize (prompt + completion) as one plain-text sequence.

    prompt_tokens is carried through so the length filter and the optional loss mask
    both key off the same number.
    """
    p = tok(example["prompt"], add_special_tokens=False)["input_ids"]
    c = tok(example["completion"], add_special_tokens=False)["input_ids"]
    ids = (p + c + [tok.eos_token_id])[:MAX_LEN]

    labels = list(ids)
    if MASK_PROMPT_LOSS:
        n = min(len(p), len(ids))
        labels[:n] = [-100] * n

    return {
        "input_ids": ids,
        "attention_mask": [1] * len(ids),
        "labels": labels,
        "prompt_tokens": len(p),
    }


split = stage_ds.map(split_dialogue, remove_columns=stage_ds.column_names)
after_split = split.filter(keep_row)
print(f"[data] marker + empty-completion filter: {len(split)} -> {len(after_split)} "
      f"({100 * len(after_split) / len(split):.1f}%)")

tokenized = after_split.map(encode, remove_columns=["completion"])   # keep prompt text for now

# The prompt must leave room for a real completion inside the context budget.
usable = tokenized.filter(lambda x: x["prompt_tokens"] <= MAX_LEN - MIN_COMPLETION_TOKENS)
print(f"[data] prompt-length filter: {len(tokenized)} -> {len(usable)} "
      f"({100 * len(usable) / len(tokenized):.1f}%)")
print(f"[data] usable yield overall: {len(usable)}/{len(stage_ds)} "
      f"({100 * len(usable) / len(stage_ds):.1f}%)")

In [ ]:
# Fixed val slice, taken off the front of this stage's half. The order is the seeded
# shuffle, so the same rows land in val on every run and the curves stay comparable.
# It comes from the SFT half, never from the half reserved for DPO.
val_ds = usable.select(range(VAL_PAIRS))
train_ds = usable.select(range(VAL_PAIRS, len(usable)))

# Prompt text for the generation sanity checks, taken from the val slice itself and
# captured before the text column is dropped.
VAL_PROMPTS = val_ds["prompt"][:N_SAMPLE_GENERATIONS]

TRAIN_PAIRS = len(train_ds)
train_ds = train_ds.remove_columns(["prompt", "prompt_tokens"])
val_ds = val_ds.remove_columns(["prompt", "prompt_tokens"])

print(f"[data] train {TRAIN_PAIRS} pairs | val {len(val_ds)} pairs (held out, fixed)")
print(f"[data] checkpoint + eval every {CKPT_EVERY_PAIRS} pairs -> "
      f"{TRAIN_PAIRS // CKPT_EVERY_PAIRS} checkpoints + final")
print(f"[data] prompt loss mask: {MASK_PROMPT_LOSS}")

## 5. Pair accounting

The counter is incremented by the collator, which sees every example that actually reaches
the model. That is what keeps it honest if batching or packing ever changes - a naive
pairs-per-step estimate would not notice.

Everything downstream hangs off that counter: when to checkpoint, when to run validation,
when to stop. Trainer's own `save_steps` is set to a plain step interval and exists only as
disconnect insurance; it is not the checkpoint cadence.

In [ ]:
import math

import wandb
from transformers import Trainer, TrainerCallback, TrainingArguments

PAIRS_PER_STEP = PER_DEVICE_BS * GRAD_ACCUM


class PairCounter:
    """Single source of truth for how many pairs the model has seen."""

    def __init__(self):
        self.n = 0


COUNTER = PairCounter()


class CountingCollator:
    """Pads a batch to its longest sequence and counts the examples it passes through.

    Counting happens here rather than off the step number so that a change to batching
    cannot silently desync the budget. Eval batches are not training pairs, so the
    cadence callback switches counting off around validation.
    """

    def __init__(self, counter, pad_id):
        self.counter = counter
        self.pad_id = pad_id
        self.counting = True

    def __call__(self, features):
        if self.counting:
            self.counter.n += len(features)

        width = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad = width - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.pad_id] * pad)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad)
            batch["labels"].append(f["labels"] + [-100] * pad)   # pads never score
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}


collator = CountingCollator(COUNTER, tok.eos_token_id)


class PairTrainer(Trainer):
    """Stamps every log with pairs_seen so train and val curves share one x-axis."""

    def log(self, logs, *args, **kwargs):
        logs = {**logs, "pairs_seen": COUNTER.n}
        return super().log(logs, *args, **kwargs)

In [ ]:
@torch.no_grad()
def sample_generations(model, prompts, max_new_tokens=64):
    """Generate one completion per prompt, batch of one to sidestep padding side.

    Loss falling while output is malformed means the preprocessing is wrong, and the
    loss curve alone will not show that.
    """
    was_training = model.training
    model.eval()
    out = []
    for prompt in prompts:
        ids = tok(prompt, return_tensors="pt", truncation=True,
                  max_length=MAX_LEN - max_new_tokens).to(model.device)
        gen = model.generate(
            **ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tok.eos_token_id,
        )
        out.append(tok.decode(gen[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
    model.train(was_training)
    return out

In [ ]:
class PairCadence(TrainerCallback):
    """Drives validation, checkpointing and stopping off the pair counter.

    Deliberately not save_steps / eval_steps / max_steps: step count moves with batch
    size, the pair budget does not.
    """

    def __init__(self, counter, collator, every, budget, ckpt_dir):
        self.counter = counter
        self.collator = collator
        self.every = every
        self.budget = budget
        self.ckpt_dir = ckpt_dir
        self.trainer = None
        self.next_boundary = every
        self.resync = False

    def on_train_begin(self, args, state, control, **kwargs):
        # On a resume the counter restarts at zero while Trainer fast-forwards the
        # dataloader through batches it has already trained on - and those go through
        # the collator. Resync to the step count once, at the first real step.
        self.resync = state.global_step > 0
        self.counter.n = state.global_step * PAIRS_PER_STEP
        self.next_boundary = (self.counter.n // self.every + 1) * self.every

        if wandb.run is not None:
            wandb.define_metric("pairs_seen")
            wandb.define_metric("*", step_metric="pairs_seen")
            wandb.config.update(
                {
                    "stage": "sft-preferred-ft",
                    "data_seed": DATA_SEED,
                    "sft_rows": list(SFT_ROWS),
                    "dpo_reserved_rows": list(DPO_ROWS),
                    "usable_pairs": len(usable),
                    "train_pairs": self.budget,
                    "val_pairs": len(val_ds),
                    "max_len": MAX_LEN,
                    "mask_prompt_loss": MASK_PROMPT_LOSS,
                    "ckpt_every_pairs": self.every,
                },
                allow_val_change=True,
            )

    def on_step_end(self, args, state, control, **kwargs):
        if self.resync:
            self.counter.n = state.global_step * PAIRS_PER_STEP
            self.resync = False

        expected = state.global_step * PAIRS_PER_STEP
        if abs(self.counter.n - expected) > PER_DEVICE_BS:
            print(f"[warn] pair counter {self.counter.n} vs {expected} expected at step "
                  f"{state.global_step} - batching or packing changed, check the accounting")

        while self.counter.n >= self.next_boundary and self.next_boundary <= self.budget:
            self._checkpoint(f"{self.next_boundary:06d}")
            self.next_boundary += self.every

        if self.counter.n >= self.budget:
            control.should_training_stop = True
        return control

    def _checkpoint(self, tag):
        seen = self.counter.n
        model = self.trainer.model

        # Eval examples are not training pairs.
        self.collator.counting = False
        try:
            metrics = self.trainer.evaluate()
        finally:
            self.collator.counting = True

        loss = metrics["eval_loss"]
        ppl = math.exp(min(loss, 20))      # guard against overflow on a broken run
        self.trainer.log({"eval_perplexity": ppl})
        print(f"[ckpt] pairs={seen} val_loss={loss:.4f} ppl={ppl:.2f}")

        path = f"{self.ckpt_dir}/pairs-{tag}"
        self.trainer.save_model(path)
        tok.save_pretrained(path)

        samples = sample_generations(model, VAL_PROMPTS)
        for prompt, text in zip(VAL_PROMPTS, samples):
            print(f"  prompt ...{prompt[-90:]!r}")
            print(f"  gen    {text[:200]!r}\n")
        if wandb.run is not None:
            table = wandb.Table(columns=["pairs_seen", "prompt", "generation"])
            for prompt, text in zip(VAL_PROMPTS, samples):
                table.add_data(seen, prompt[-400:], text)
            wandb.log({"samples": table, "pairs_seen": seen})

## 6. Train

`report_to=["wandb"]` lets Trainer own `wandb.init` - calling it by hand as well splits the
run in two.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tok.eos_token_id

# transformers 5.x dropped warmup_ratio; an explicit step count works on 4.x and 5.x alike.
TOTAL_STEPS = math.ceil(TRAIN_PAIRS / PAIRS_PER_STEP)
WARMUP_STEPS = math.ceil(WARMUP_RATIO * TOTAL_STEPS)

args = TrainingArguments(
    output_dir=RESUME_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=16,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),    # a T4 is Turing: fp16, not bf16
    logging_steps=LOG_EVERY_STEPS,

    # Validation and the real checkpoints are driven by the pair counter, not by these.
    eval_strategy="no",
    save_strategy="steps",             # rolling Trainer state for disconnect recovery only
    save_steps=100,
    save_total_limit=1,

    dataloader_num_workers=0,          # keeps the collator in lockstep with the step
    report_to=["wandb"],
    run_name=RUN_NAME,
    seed=DATA_SEED,
)

cadence = PairCadence(COUNTER, collator, CKPT_EVERY_PAIRS, TRAIN_PAIRS, CKPT_DIR)

kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[cadence],
)
try:                                   # transformers renamed this argument
    trainer = PairTrainer(**kwargs, processing_class=tok)
except TypeError:
    trainer = PairTrainer(**kwargs, tokenizer=tok)

cadence.trainer = trainer
print(f"[train] {TRAIN_PAIRS} pairs, {PAIRS_PER_STEP} pairs/step, "
      f"~{TOTAL_STEPS} steps, {WARMUP_STEPS} warmup")

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

RESUME = False        # flip to True after a disconnect, then re-run this cell

resume_from = get_last_checkpoint(RESUME_DIR) if RESUME else None
if RESUME:
    print(f"[train] resuming from {resume_from}")

trainer.train(resume_from_checkpoint=resume_from)

## 7. Final checkpoint and manifest

The manifest records the seed and the row range so the DPO stage can reconstruct which half
this model saw, and can hold out the other half for itself.

In [ ]:
import json

trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)

manifest = {
    "stage": "sft-preferred-ft",
    "model": MODEL_NAME,
    "dataset": f"{DATASET}:{DATA_DIR}",
    "split": "train",
    "sampling": "ds.shuffle(seed=DATA_SEED).select(range(*sft_rows))",
    "data_seed": DATA_SEED,
    "raw_rows": RAW_N,
    "sft_rows": list(SFT_ROWS),
    "dpo_reserved_rows": list(DPO_ROWS),
    "usable_pairs": len(usable),
    "val_pairs": len(val_ds),
    "train_pairs": TRAIN_PAIRS,
    "pairs_seen": COUNTER.n,
    "max_len": MAX_LEN,
    "mask_prompt_loss": MASK_PROMPT_LOSS,
    "ckpt_every_pairs": CKPT_EVERY_PAIRS,
    "wandb": {"project": "DPO-SAE", "run_id": RUN_NAME},
}

with open(f"{RUN_DIR}/manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)

print(json.dumps(manifest, indent=2))
print(f"\n[done] final checkpoint -> {FINAL_DIR}")

In [ ]:
# Final sanity check: does it answer like an assistant turn?
for prompt, text in zip(VAL_PROMPTS, sample_generations(trainer.model, VAL_PROMPTS)):
    print(f"prompt ...{prompt[-120:]!r}")
    print(f"gen    {text!r}\n")

## 8. Before calling the run good

- **Train and val loss should not diverge.** One epoch at this budget is enough; if they
  split, stop. Do not add epochs to use the data better - a model that memorizes the
  `chosen` completions shrinks the logprob gap DPO operates on.
- **Read the generations, not just the loss.** The point of this stage is dialogue format.
  Falling loss with malformed output means the preprocessing broke.
- **Check the usable yield.** A sudden drop from the ~99% range means the splitter broke.
- **Watch Drive quota.** Five checkpoints plus the final is roughly 3GB. Once the run is
  validated the intermediates can be pruned, but keep `final/`.
- **Hand the DPO stage `manifest.json`.** It needs `data_seed` and `dpo_reserved_rows` to
  train on the half this model never saw.